# Convert EPS to PDF


In [ ]:
#!/usr/bin/env python3
from pathlib import Path
import argparse
import shutil
import subprocess
import sys

# python kernel selection
# Cmd + P
# input: >Python: Select Interpreter

def find_eps_converter() -> tuple[str, str]:
    """Find an EPS-to-PDF converter."""
    epstopdf = shutil.which("epstopdf")
    ps2pdf = shutil.which("ps2pdf")

    if epstopdf:
        return "epstopdf", epstopdf

    if ps2pdf:
        return "ps2pdf", ps2pdf

    raise FileNotFoundError(
        "Cannot find epstopdf or ps2pdf. Install Ghostscript or TeX Live/MacTeX first."
    )

def convert_eps_to_pdf(eps_path: Path, overwrite: bool = False) -> None:
    eps_path = eps_path.resolve()
    pdf_path = eps_path.with_suffix(".pdf")

    if pdf_path.exists() and not overwrite:
        print(f"Skip existing: {pdf_path}")
        return

    converter_name, converter_path = find_eps_converter()

    if converter_name == "epstopdf":
        command = [
            converter_path,
            str(eps_path),
            f"--outfile={pdf_path}",
        ]
    else:
        command = [
            converter_path,
            "-dEPSCrop",
            str(eps_path),
            str(pdf_path),
        ]

    print(f"Converting: {eps_path} -> {pdf_path}")
    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Failed to convert {eps_path}")

    print(f"Done: {pdf_path}")

def collect_eps_files(path: Path, recursive: bool = False) -> list[Path]:
    path = path.resolve()

    if path.is_file():
        if path.suffix.lower() != ".eps":
            raise ValueError(f"Input file is not an EPS file: {path}")
        return [path]

    if path.is_dir():
        pattern = "**/*.eps" if recursive else "*.eps"
        return sorted(path.glob(pattern))

    raise FileNotFoundError(f"Path does not exist: {path}")

def main() -> None:
    parser = argparse.ArgumentParser(
        description="Convert EPS files to PDF without rasterizing."
    )
    parser.add_argument(
        "path",
        nargs="?",
        default=".",
        help="EPS file or directory containing EPS files. Default: current directory.",
    )
    parser.add_argument(
        "-r", "--recursive",
        action="store_true",
        help="Search EPS files recursively in subdirectories.",
    )
    parser.add_argument(
        "-f", "--overwrite",
        action="store_true",
        help="Overwrite existing PDF files.",
    )

    args, unknown = parser.parse_known_args()

    eps_files = collect_eps_files(Path(args.path), recursive=args.recursive)

    if not eps_files:
        print("No EPS files found.")
        return

    for eps_file in eps_files:
        convert_eps_to_pdf(eps_file, overwrite=args.overwrite)

if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        print(f"Error: {exc}", file=sys.stderr)
        sys.exit(1)


Converting: /Users/lu/Repos/Thesis GitHub/figures_proj2/proj2.6.eps -> /Users/lu/Repos/Thesis GitHub/figures_proj2/proj2.6.pdf
Done: /Users/lu/Repos/Thesis GitHub/figures_proj2/proj2.6.pdf
Converting: /Users/lu/Repos/Thesis GitHub/figures_proj2/proj2.8.eps -> /Users/lu/Repos/Thesis GitHub/figures_proj2/proj2.8.pdf
Done: /Users/lu/Repos/Thesis GitHub/figures_proj2/proj2.8.pdf
